# 🏆 Sports 100 Classes: PyTorch (EfficientNet-B0)

Обучение классификатора 100 видов спорта и сохранение файлов для FastAPI.

**Перед запуском:**
1. `Runtime → Change runtime type → T4 GPU`.
2. Загрузите `archive.zip` (датасет) в **Google Drive → Мой диск** (`My Drive/archive.zip`) или прямо в Colab в папку `/content` (значок папки слева).
3. `Runtime → Run all`.

В конце скачаются 2 файла: `pytorch_sports_model.pth` и `classes.json`. Их нужно положить в папку `model` проекта.

In [ ]:
# 1. Проверка GPU
import torch
print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ GPU нет: обучение будет очень медленным. Runtime → Change runtime type → T4 GPU")

In [ ]:
# 2. Настройки
ZIP_PATH = "/content/archive.zip"            # если загрузили zip прямо в Colab
DRIVE_ZIP_PATH = "/content/drive/MyDrive/archive.zip"  # если лежит на Google Drive
DATA = "/content/extracted_data"             # сюда распаковываем
EPOCHS = 5
BATCH_SIZE = 64
NUM_WORKERS = 2
PRETRAINED = True                            # веса ImageNet для EfficientNet-B0
MODEL_OUT = "/content/pytorch_sports_model.pth"
CLASSES_OUT = "/content/classes.json"

In [ ]:
# 3. Распаковка датасета (файл .h5 внутри архива нам не нужен)
import os, zipfile

if not os.path.isdir(os.path.join(DATA, "train")):
    if not os.path.exists(ZIP_PATH):
        from google.colab import drive
        drive.mount("/content/drive")
        ZIP_PATH = DRIVE_ZIP_PATH
    assert os.path.exists(ZIP_PATH), f"Не найден архив: {ZIP_PATH}"
    with zipfile.ZipFile(ZIP_PATH) as z:
        members = [m for m in z.namelist() if not m.lower().endswith(".h5")]
        z.extractall(DATA, members)
print("Папки:", sorted(os.listdir(DATA)))

In [ ]:
# 4. Данные
import json, time
import torch.nn as nn
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = datasets.ImageFolder(os.path.join(DATA, "train"), train_tf)
val_ds = datasets.ImageFolder(os.path.join(DATA, "valid"), eval_tf)
test_ds = datasets.ImageFolder(os.path.join(DATA, "test"), eval_tf)
NUM_CLASSES = len(train_ds.classes)
print(f"Классов: {NUM_CLASSES} | train: {len(train_ds)} | valid: {len(val_ds)} | test: {len(test_ds)}")

kw = dict(num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, **kw)
val_dl = DataLoader(val_ds, batch_size=128, **kw)
test_dl = DataLoader(test_ds, batch_size=128, **kw)

In [ ]:
# 5. Модель (такая же архитектура в fastapi_app.py)
class SportsClassifier(nn.Module):
    def __init__(self, num_classes=100, pretrained=True):
        super().__init__()
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        self.backbone = models.efficientnet_b0(weights=weights)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier[1] = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)

model = SportsClassifier(NUM_CLASSES, pretrained=PRETRAINED).to(device)
print("Параметров:", sum(p.numel() for p in model.parameters()))

In [ ]:
# 6. Обучение
use_amp = device.type == "cuda"
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=1e-3, total_steps=EPOCHS * len(train_dl))
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)


def evaluate(loader):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out = model(x)
                loss_sum += criterion(out, y).item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            total += len(y)
    return loss_sum / total, correct / total


history = {"train_loss": [], "val_loss": [], "val_acc": []}
best_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0, run_loss, seen = time.time(), 0.0, 0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        run_loss += loss.item() * len(y)
        seen += len(y)
    val_loss, val_acc = evaluate(val_dl)
    history["train_loss"].append(run_loss / seen)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    print(f"Epoch {epoch}/{EPOCHS} | train_loss {run_loss / seen:.4f} | val_loss {val_loss:.4f} | "
          f"val_acc {val_acc:.2%} | {time.time() - t0:.0f} c")
    if val_acc >= best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), MODEL_OUT)  # сохраняем лучшую по valid

print(f"Лучшая точность на valid: {best_acc:.2%}")

In [ ]:
# 7. Итог на тестовой выборке + графики (сделайте скриншот этой ячейки)
import matplotlib.pyplot as plt

model.load_state_dict(torch.load(MODEL_OUT, map_location=device))
test_loss, test_acc = evaluate(test_dl)
print(f"✅ TEST ACCURACY: {test_acc:.2%}  ({len(test_ds)} изображений, {NUM_CLASSES} классов)")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(history["train_loss"], label="train"); ax[0].plot(history["val_loss"], label="valid")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(history["val_acc"], marker="o"); ax[1].set_title("Valid accuracy"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

In [ ]:
# 8. Сохранение списка классов (порядок = индексы модели) и скачивание файлов
with open(CLASSES_OUT, "w", encoding="utf-8") as f:
    json.dump(train_ds.classes, f, ensure_ascii=False, indent=2)
print("Сохранено:", MODEL_OUT, "и", CLASSES_OUT)
print("Размер модели: %.1f МБ" % (os.path.getsize(MODEL_OUT) / 1e6))

from google.colab import files
files.download(MODEL_OUT)
files.download(CLASSES_OUT)

In [ ]:
# 9. Проверка на картинках из test (скриншот «текшеруу»)
import random
import matplotlib.pyplot as plt

classes = train_ds.classes
model.eval()
samples = random.sample(range(len(test_ds)), 8)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, idx in zip(axes.flat, samples):
    path, true_idx = test_ds.samples[idx]
    img = test_ds[idx][0].unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(img), dim=1)[0]
    pred = int(probs.argmax())
    ax.imshow(plt.imread(path))
    ok = pred == true_idx
    ax.set_title(f"{classes[pred]} ({probs[pred]:.0%})\nистина: {classes[true_idx]}", fontsize=9,
                 color="green" if ok else "red")
    ax.axis("off")
plt.tight_layout(); plt.show()